# ACG — optimisation corrigée, 13 jeux réels + 4 synthétiques

Benchmark de régression à budget de paramètres contrôlé :

- **13 jeux réels** identiques à Kumar et al. (TMLR 2026), chargés depuis leur dépôt officiel ;
- **4 jeux synthétiques** : ACG, CP, TT et Friedman-1 ;
- **3 optimiseurs ACG à architecture strictement identique** :
  `ACG-BCD-Ball`, `ACG-GD-Free`, `ACG-SBG-Riemannian` ;
- **6 baselines** : CP-spline, TT-spline, Spline-KAN, MLP, ResNet et CatBoost ;
- 5 folds identiques, 3 initialisations par fold ;
- prétraitement ajusté uniquement sur le sous-ensemble d'entraînement ;
- BCD coordonnée corrigé : un bloc $Q_k$, un bloc $\Theta_s$, puis le readout ;
- projection de $\Theta$ sur une **boule** pour la variante BCD ;
- descente jointe avec $\Theta$ libre ;
- block-gradient stochastique avec gradient riemannien et $\Theta$ sur un produit de sphères ;
- `torch.compile(mode="max-autotune")` pour les méthodes gradientielles ; le BCD
  reste en eager car il construit dynamiquement les designs affines exacts ;
- NRMSE, $R^2$, paramètres actifs, temps et mémoire GPU ;
- rangs moyens, Friedman, Nemenyi et Wilcoxon–Holm ;
- reprise automatique et export CSV/LaTeX/PDF/PNG/ZIP.

> **Protocole confirmatoire.** La comparaison principale utilise uniquement
> `ACG-BCD-Ball` face aux six baselines. Les deux autres variantes ACG forment une
> ablation d'optimisation pré-déclarée ; elles ne servent pas à sélectionner après
> coup le meilleur ACG sur le test.

> **Égalité du nombre de paramètres.** Les trois variantes ACG ont exactement les
> mêmes paramètres, le même $m$, le même rang, les mêmes folds et initialisations.
> Pour les autres architectures, les hyperparamètres entiers sont choisis au plus
> près du budget ACG et l'écart actif exact est publié. CatBoost utilise un budget
> structurel (feuilles + seuils).

In [ ]:
#@title 0. Installation, stockage et données TMLR
!pip -q install "pandas>=2.2" "scikit-learn>=1.5" "scipy>=1.12" \
    "statsmodels>=0.14" "scikit-posthocs>=0.9" "seaborn>=0.13" \
    "catboost>=1.2.7" "nvidia-ml-py>=12.560" "nbformat>=5.10"

import os
import subprocess
from pathlib import Path

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = Path("/content/drive/MyDrive/ACG_TMLR_Benchmark")
else:
    WORK_ROOT = Path("/content/ACG_TMLR_Benchmark")

WORK_ROOT.mkdir(parents=True, exist_ok=True)
DATA_REPO = Path("/content/tdl-numerical-encodings")
if not DATA_REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/mkumar73/tdl-numerical-encodings.git",
        str(DATA_REPO),
    ], check=True)

DATA_COMMIT = subprocess.check_output(
    ["git", "-C", str(DATA_REPO), "rev-parse", "HEAD"], text=True
).strip()
print("Répertoire de travail :", WORK_ROOT)
print("Commit des données     :", DATA_COMMIT)

## 1. Configuration

Le profil `paper` exécute le protocole demandé. Il représente
$13\times5\times3\times9=1755$ ajustements réels, plus les expériences
synthétiques. Activez Google Drive : chaque ajustement et chaque courbe de
convergence sont enregistrés immédiatement et la reprise est automatique.

In [ ]:
#@title 1. Configuration de l'expérience
import copy
import gc
import hashlib
import json
import math
import random
import shutil
import threading
import time
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from catboost import CatBoostRegressor
from IPython.display import display
from scipy.stats import friedmanchisquare, wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from statsmodels.stats.multitest import multipletests
import scikit_posthocs as sp

warnings.filterwarnings("ignore", category=UserWarning)

if not torch.cuda.is_available():
    raise RuntimeError("Activez un GPU dans Colab : Runtime > Change runtime type > GPU.")

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

PROFILE = "paper"  #@param ["smoke", "paper", "extended"]
PROFILES = {
    "smoke": {
        "folds": 2, "seeds": [0], "gradient_cycles": 30, "bcd_cycles": 3,
        "eval_every_cycles": 5, "patience": 3, "batch_size": 512,
        "bcd_chunk_size": 1024, "theta_inner_steps": 12,
        "real_limit": 2, "synthetic_limit": 1,
    },
    "paper": {
        "folds": 5, "seeds": [0, 1, 2], "gradient_cycles": 400, "bcd_cycles": 20,
        "eval_every_cycles": 10, "patience": 7, "batch_size": 1024,
        "bcd_chunk_size": 8192, "theta_inner_steps": 30,
        "real_limit": None, "synthetic_limit": None,
    },
    "extended": {
        "folds": 5, "seeds": [0, 1, 2, 3, 4], "gradient_cycles": 800, "bcd_cycles": 40,
        "eval_every_cycles": 10, "patience": 12, "batch_size": 1024,
        "bcd_chunk_size": 8192, "theta_inner_steps": 50,
        "real_limit": None, "synthetic_limit": None,
    },
}
CFG = PROFILES[PROFILE]

REAL_DATASETS = [
    "abalone", "ca_housing", "cpu_small", "diamonds", "house_sales",
    "parkinsons", "wine_quality_reg", "house8L", "pulsar", "sulphur",
    "fifa_wage", "sgemm_gpu", "protein",
]
SYNTHETIC_DATASETS = [
    "ACG-structured", "CP-structured", "TT-structured", "Friedman-1"
]
PRIMARY_ACG = "ACG-BCD-Ball"
ACG_VARIANTS = ["ACG-BCD-Ball", "ACG-GD-Free", "ACG-SBG-Riemannian"]
BASELINES = ["CP-spline", "TT-spline", "Spline-KAN", "MLP", "ResNet", "CatBoost"]
MODEL_NAMES = ACG_VARIANTS + BASELINES
CONFIRMATORY_MODELS = [PRIMARY_ACG] + BASELINES

N_BASIS = 7
SPLINE_DEGREE = 3
ACG_M = 4
ACG_RANK = 2
THETA_RADIUS = 1.0
SYNTHETIC_N = 6000
INNER_VAL_SIZE = 0.10
LR_EUCLIDEAN = 5e-3
LR_THETA = 2e-2
WEIGHT_DECAY = 1e-5
BCD_RIDGE = 1e-4
BCD_PROX = 1e-5
GRAD_CLIP = 5.0
USE_TORCH_COMPILE = True
TORCH_COMPILE_MODE = "max-autotune"
RESUME = True

OUTDIR = WORK_ROOT / f"results_{PROFILE}_optimization_v2"
OUTDIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = OUTDIR / "raw_results.csv"
ERROR_PATH = OUTDIR / "errors.csv"
HISTORY_PATH = OUTDIR / "optimization_history.csv"

print("GPU     :", GPU_NAME)
print("Profil  :", PROFILE, CFG)
print("Sorties :", OUTDIR)

## 2. Données et prétraitement sans fuite

Les CSV sont ceux du dépôt officiel de l'article TMLR. Dans chaque répétition :

1. le fold test est fixé une fois et partagé par tous les modèles ;
2. 10 % du fold d'apprentissage sert à l'arrêt anticipé ;
3. `MinMaxScaler` et `OneHotEncoder` sont ajustés uniquement sur la partie `fit` ;
4. la moyenne et l'écart-type de la cible sont calculés uniquement sur `fit`.

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@dataclass
class TabularDataset:
    name: str
    kind: str
    X: pd.DataFrame
    y: np.ndarray


@dataclass
class PreparedFold:
    x_fit: np.ndarray
    x_val: np.ndarray
    x_test: np.ndarray
    y_fit: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray
    y_mean: float
    y_std: float
    feature_names: list[str]


def load_real_dataset(name: str) -> TabularDataset:
    path = DATA_REPO / "datasets" / "regression" / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed:")]
    if "Targets" not in df.columns:
        raise ValueError(f"Colonne Targets absente dans {path}")
    df = df.replace([np.inf, -np.inf], np.nan).dropna(axis=0).reset_index(drop=True)
    y = pd.to_numeric(df.pop("Targets"), errors="raise").to_numpy(np.float32)
    return TabularDataset(name=name, kind="real", X=df, y=y)


def make_preprocessor(X_fit: pd.DataFrame) -> ColumnTransformer:
    numerical = X_fit.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical = [c for c in X_fit.columns if c not in numerical]
    transformers = []
    if numerical:
        transformers.append(("num", MinMaxScaler(clip=True), numerical))
    if categorical:
        transformers.append((
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32),
            categorical,
        ))
    return ColumnTransformer(transformers, remainder="drop", sparse_threshold=0.0)


def prepare_fold(
    dataset: TabularDataset,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    seed: int,
) -> PreparedFold:
    fit_idx, val_idx = train_test_split(
        train_idx, test_size=INNER_VAL_SIZE, random_state=10_000 + seed, shuffle=True
    )
    X_fit = dataset.X.iloc[fit_idx]
    X_val = dataset.X.iloc[val_idx]
    X_test = dataset.X.iloc[test_idx]
    pre = make_preprocessor(X_fit)
    x_fit = np.asarray(pre.fit_transform(X_fit), dtype=np.float32)
    x_val = np.asarray(pre.transform(X_val), dtype=np.float32)
    x_test = np.asarray(pre.transform(X_test), dtype=np.float32)
    feature_names = pre.get_feature_names_out().tolist()

    y_fit_raw = dataset.y[fit_idx].astype(np.float32)
    y_val_raw = dataset.y[val_idx].astype(np.float32)
    y_test_raw = dataset.y[test_idx].astype(np.float32)
    y_mean = float(y_fit_raw.mean())
    y_std = float(y_fit_raw.std())
    y_std = max(y_std, 1e-8)
    return PreparedFold(
        x_fit=x_fit, x_val=x_val, x_test=x_test,
        y_fit=(y_fit_raw - y_mean) / y_std,
        y_val=(y_val_raw - y_mean) / y_std,
        y_test=y_test_raw,
        y_mean=y_mean, y_std=y_std,
        feature_names=feature_names,
    )


def regression_metrics(y_true: np.ndarray, pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.float64)
    pred = np.asarray(pred, dtype=np.float64)
    mse = float(np.mean((pred - y_true) ** 2))
    var = max(float(np.mean((y_true - y_true.mean()) ** 2)), 1e-12)
    return {
        "rmse": math.sqrt(mse),
        "nrmse": math.sqrt(mse / var),
        "r2": 1.0 - mse / var,
    }

## 3. Bases B-splines

In [ ]:
EPS = 1e-8


def open_uniform_knots(n_basis: int, degree: int, device, dtype) -> torch.Tensor:
    if n_basis < degree + 1:
        raise ValueError("n_basis doit être >= degree + 1")
    n_internal = n_basis - degree - 1
    internal = (
        torch.linspace(0.0, 1.0, n_internal + 2, device=device, dtype=dtype)[1:-1]
        if n_internal > 0 else torch.empty(0, device=device, dtype=dtype)
    )
    return torch.cat([
        torch.zeros(degree + 1, device=device, dtype=dtype),
        internal,
        torch.ones(degree + 1, device=device, dtype=dtype),
    ])


def bspline_basis_1d(x: torch.Tensor, n_basis: int, degree: int = 3) -> torch.Tensor:
    x = x.clamp(0.0, 1.0)
    knots = open_uniform_knots(n_basis, degree, x.device, x.dtype)
    n0 = knots.numel() - 1
    basis = ((x[..., None] >= knots[:-1]) & (x[..., None] < knots[1:])).to(x.dtype)
    for q in range(1, degree + 1):
        n_curr = n0 - q
        left_den = knots[q:q+n_curr] - knots[:n_curr]
        right_den = knots[q+1:q+1+n_curr] - knots[1:1+n_curr]
        left = torch.where(
            left_den > 0,
            (x[..., None] - knots[:n_curr]) / left_den.clamp_min(EPS),
            torch.zeros_like(left_den),
        )
        right = torch.where(
            right_den > 0,
            (knots[q+1:q+1+n_curr] - x[..., None]) / right_den.clamp_min(EPS),
            torch.zeros_like(right_den),
        )
        basis = left * basis[..., :n_curr] + right * basis[..., 1:n_curr+1]
    endpoint = F.one_hot(
        torch.full_like(x, n_basis - 1, dtype=torch.long), n_basis
    ).to(x.dtype)
    return torch.where((x >= 1.0 - 1e-7)[..., None], endpoint, basis)


def feature_basis(x: torch.Tensor) -> torch.Tensor:
    return bspline_basis_1d(x, N_BASIS, SPLINE_DEGREE)


_check_x = torch.linspace(0, 1, 257)
_check_b = bspline_basis_1d(_check_x, N_BASIS, SPLINE_DEGREE)
assert _check_b.shape == (257, N_BASIS)
assert torch.allclose(_check_b.sum(-1), torch.ones_like(_check_x), atol=2e-5)
print("B-splines validées :", tuple(_check_b.shape))

## 4. $\Theta$-algèbre : libre, boule et sphère

Les trois optimiseurs partagent exactement la même loi de produit et le même
nombre de paramètres. Seul le domaine d'optimisation de $\Theta$ change :

\[
\Theta_{\rm BCD}\in\prod_{s,h}\{u:\|u\|_2\le r\},\qquad
\Theta_{\rm GD}\in\mathbb R^{(p-1)\times H\times(m-1)},
\]

et, pour l'ablation riemannienne,

\[
\Theta_{\rm Riem}\in\prod_{s,h}\mathbb S^{m-2}(r),\quad
\operatorname{grad}f=g-\frac{\langle g,\theta\rangle}{r^2}\theta.
\]

La boule inclut $\Theta=0$ et permet donc à l'optimisation BCD d'éteindre une
interaction antisymétrique inutile, contrairement à la sphère.

In [ ]:
def pair_indices(d: int, device=None) -> tuple[torch.Tensor, torch.Tensor]:
    ij = torch.triu_indices(d, d, offset=1, device=device)
    return ij[0], ij[1]


def quaternion_theta(device, dtype) -> torch.Tensor:
    theta = torch.zeros(3, 3, device=device, dtype=dtype)
    theta[0, 2] = 1.0
    theta[1, 1] = -1.0
    theta[2, 0] = 1.0
    return theta


def retract_product_sphere_(theta: torch.Tensor, radius: float) -> None:
    with torch.no_grad():
        theta.mul_(radius / theta.norm(dim=-1, keepdim=True).clamp_min(EPS))


def project_product_ball_(theta: torch.Tensor, radius: float) -> None:
    with torch.no_grad():
        norms = theta.norm(dim=-1, keepdim=True).clamp_min(EPS)
        theta.mul_(torch.clamp(radius / norms, max=1.0))


def riemannian_sphere_gradient(
    theta: torch.Tensor, grad: torch.Tensor, radius: float
) -> torch.Tensor:
    radial = (grad * theta).sum(dim=-1, keepdim=True) / (radius ** 2)
    return grad - radial * theta


def theta_product(x, y, theta, pair_a, pair_b):
    x0, xv = x[..., :1], x[..., 1:]
    y0, yv = y[..., :1], y[..., 1:]
    scalar = x0 * y0 - (xv * yv).sum(dim=-1, keepdim=True)
    wedge = xv[..., pair_a] * yv[..., pair_b] - xv[..., pair_b] * yv[..., pair_a]
    cross = torch.einsum("...h,hd->...d", wedge, theta)
    vector = x0 * yv + y0 * xv + cross
    return torch.cat([scalar, vector], dim=-1)

## 5. Modèles différentiables

Chaque modèle expose des blocs logiques pour le BCD. Les modèles utilisant des splines reçoivent exactement la même base cubique à 7 fonctions par variable.

In [ ]:
@dataclass
class BlockSpec:
    name: str
    params: list[nn.Parameter]
    geometry: str = "euclidean"


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


class ACGRegressor(nn.Module):
    input_kind = "basis"

    def __init__(
        self, p, n_basis=N_BASIS, m=ACG_M, rank=ACG_RANK,
        radius=THETA_RADIUS, theta_mode="ball",
    ):
        super().__init__()
        self.p, self.n_basis, self.m, self.rank = p, n_basis, m, rank
        self.theta_radius = radius
        self.theta_mode = theta_mode
        d = m - 1
        pa, pb = pair_indices(d)
        self.register_buffer("pair_a", pa)
        self.register_buffer("pair_b", pb)
        q = torch.randn(rank, p, n_basis, m) * (0.16 / math.sqrt(m))
        q[..., 0] += 1.0
        self.q = nn.Parameter(q)
        self.rho = nn.Parameter(torch.randn(rank, m) * 0.10)
        self.alpha = nn.Parameter(torch.zeros(()))
        if m == 4:
            theta = quaternion_theta("cpu", torch.float32)[None].repeat(p - 1, 1, 1)
            theta = theta + 0.03 * torch.randn_like(theta)
        else:
            theta = torch.randn(p - 1, pa.numel(), d)
        self.theta = nn.Parameter(theta)
        if theta_mode == "sphere":
            retract_product_sphere_(self.theta, self.theta_radius)
        elif theta_mode == "ball":
            project_product_ball_(self.theta, self.theta_radius)
        elif theta_mode != "free":
            raise ValueError(f"theta_mode inconnu: {theta_mode}")

    def compose_lifts(self, lifts, theta=None, return_states=False):
        theta = self.theta if theta is None else theta
        state = lifts[:, :, 0]
        states_before, states_after = [], []
        for s in range(self.p - 1):
            states_before.append(state)
            state = theta_product(
                state, lifts[:, :, s + 1], theta[s], self.pair_a, self.pair_b
            )
            states_after.append(state)
        if return_states:
            return state, states_before, states_after
        return state

    def forward(self, basis):
        lifts = torch.einsum("bpn,rpnm->brpm", basis, self.q)
        state = self.compose_lifts(lifts)
        return self.alpha + torch.einsum("brm,rm->b", state, self.rho)

    def coarse_parameter_blocks(self):
        return [
            BlockSpec("lifts_q", [self.q]),
            BlockSpec("theta_manifold", [self.theta], "sphere"),
            BlockSpec("readout", [self.rho, self.alpha]),
        ]

    def parameter_blocks(self):
        return self.coarse_parameter_blocks()

    @torch.no_grad()
    def gauge_balance_(self):
        # Pour chaque chemin, égalise les normes des Q_k. Le produit des facteurs
        # de remise à l'échelle vaut un : la fonction représentée ne change pas.
        norms = self.q.flatten(2).norm(dim=-1).clamp_min(1e-8)
        geom = norms.log().mean(dim=1, keepdim=True).exp()
        self.q.mul_((geom / norms)[:, :, None, None])


class CPRegressor(nn.Module):
    input_kind = "basis"

    def __init__(self, p, n_basis=N_BASIS, rank=8):
        super().__init__()
        self.a = nn.Parameter(torch.randn(rank, p, n_basis) * 0.06 + 1.0)
        self.lam = nn.Parameter(torch.randn(rank) * 0.10)
        self.alpha = nn.Parameter(torch.zeros(()))

    def forward(self, basis):
        values = torch.einsum("bpn,rpn->brp", basis, self.a)
        return self.alpha + torch.einsum("br,r->b", values.prod(-1), self.lam)

    def parameter_blocks(self):
        return [BlockSpec("cp_factors", [self.a]), BlockSpec("readout", [self.lam, self.alpha])]


class TTRegressor(nn.Module):
    input_kind = "basis"

    def __init__(self, p, n_basis=N_BASIS, ranks=None):
        super().__init__()
        if ranks is None:
            ranks = [1] + [3] * (p - 1) + [1]
        self.ranks = list(ranks)
        cores = []
        for k in range(p):
            r0, r1 = ranks[k], ranks[k + 1]
            core = torch.randn(n_basis, r0, r1) * (0.18 / math.sqrt(max(r0, 1)))
            if r0 == r1:
                core = core + 0.7 * torch.eye(r0)[None]
            elif r0 == 1:
                core[:, 0, 0] += 0.8
            cores.append(nn.Parameter(core))
        self.cores = nn.ParameterList(cores)
        self.alpha = nn.Parameter(torch.zeros(()))

    def forward(self, basis):
        state = torch.ones(basis.shape[0], 1, 1, device=basis.device, dtype=basis.dtype)
        for k, core in enumerate(self.cores):
            mat = torch.einsum("bn,nij->bij", basis[:, k], core)
            state = torch.bmm(state, mat)
        return self.alpha + state[:, 0, 0]

    def parameter_blocks(self):
        groups = np.array_split(np.arange(len(self.cores)), min(3, len(self.cores)))
        blocks = []
        for j, group in enumerate(groups):
            params = [self.cores[int(k)] for k in group]
            if j == len(groups) - 1:
                params = params + [self.alpha]
            blocks.append(BlockSpec(f"tt_cores_{j}", params))
        return blocks


class SplineKAN(nn.Module):
    input_kind = "basis"

    def __init__(self, p, n_basis=N_BASIS, hidden=8):
        super().__init__()
        self.n_basis = n_basis
        self.a = nn.Parameter(torch.randn(hidden, p, n_basis) * 0.07)
        self.c = nn.Parameter(torch.randn(hidden, n_basis) * 0.09)
        self.alpha = nn.Parameter(torch.zeros(()))

    def forward(self, basis):
        hidden = torch.einsum("bpn,hpn->bh", basis, self.a).tanh()
        out_basis = bspline_basis_1d(0.5 * (hidden + 1.0), self.n_basis, SPLINE_DEGREE)
        return self.alpha + torch.einsum("bhn,hn->b", out_basis, self.c)

    def parameter_blocks(self):
        return [BlockSpec("kan_inner", [self.a]), BlockSpec("kan_outer", [self.c, self.alpha])]


class MLPRegressor(nn.Module):
    input_kind = "raw"

    def __init__(self, p, width=32):
        super().__init__()
        self.l1 = nn.Linear(p, width)
        self.l2 = nn.Linear(width, width)
        self.out = nn.Linear(width, 1)

    def forward(self, x):
        x = torch.tanh(self.l1(x))
        x = torch.tanh(self.l2(x))
        return self.out(x).squeeze(-1)

    def parameter_blocks(self):
        return [
            BlockSpec("mlp_l1", list(self.l1.parameters())),
            BlockSpec("mlp_l2", list(self.l2.parameters())),
            BlockSpec("mlp_out", list(self.out.parameters())),
        ]


class ResidualBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm = nn.LayerNorm(width)
        self.l1 = nn.Linear(width, width)
        self.l2 = nn.Linear(width, width)

    def forward(self, x):
        h = self.l2(F.gelu(self.l1(self.norm(x))))
        return x + h / math.sqrt(2.0)


class ResNetRegressor(nn.Module):
    input_kind = "raw"

    def __init__(self, p, width=24, n_blocks=1):
        super().__init__()
        self.stem = nn.Linear(p, width)
        self.blocks = nn.ModuleList([ResidualBlock(width) for _ in range(n_blocks)])
        self.out = nn.Linear(width, 1)

    def forward(self, x):
        h = F.gelu(self.stem(x))
        for block in self.blocks:
            h = block(h)
        return self.out(h).squeeze(-1)

    def parameter_blocks(self):
        middle = [p for block in self.blocks for p in block.parameters()]
        return [
            BlockSpec("resnet_stem", list(self.stem.parameters())),
            BlockSpec("resnet_blocks", middle),
            BlockSpec("resnet_out", list(self.out.parameters())),
        ]

## 6. Appariement automatique du budget

Le budget cible est le nombre de paramètres actifs de l'ACG avec $m=4$, $R=2$ et 7 B-splines. Pour les modèles différentiables, la recherche porte sur le rang, la largeur ou le nombre de blocs. Pour le TT, un vecteur de rangs non uniforme permet un appariement plus fin.

In [ ]:
def choose_nearest(candidates, budget):
    return min(candidates, key=lambda item: (abs(item[1] - budget), item[1] > budget, item[1]))


def tt_param_count(p, n_basis, ranks):
    return int(sum(n_basis * ranks[k] * ranks[k + 1] for k in range(p)) + 1)


def matched_tt_ranks(p, n_basis, budget):
    ranks = [1] * (p + 1)
    best = (list(ranks), tt_param_count(p, n_basis, ranks))
    for _ in range(10_000):
        trials = []
        for k in range(1, p):
            candidate = list(ranks)
            candidate[k] += 1
            count = tt_param_count(p, n_basis, candidate)
            trials.append((candidate, count))
            if abs(count - budget) < abs(best[1] - budget):
                best = (candidate, count)
        below = [item for item in trials if item[1] <= budget]
        if not below:
            break
        next_item = max(below, key=lambda item: item[1])
        if next_item[1] <= tt_param_count(p, n_basis, ranks):
            break
        ranks = next_item[0]
        if ranks == best[0] and best[1] == budget:
            break
    return best[0]


def acg_budget(p):
    return count_parameters(ACGRegressor(p, theta_mode="ball"))


def build_budget_matched_model(name, p, budget):
    if name in ACG_VARIANTS:
        theta_mode = {
            "ACG-BCD-Ball": "ball",
            "ACG-GD-Free": "free",
            "ACG-SBG-Riemannian": "sphere",
        }[name]
        model = ACGRegressor(p, theta_mode=theta_mode)
        config = {
            "m": ACG_M, "rank": ACG_RANK, "theta_mode": theta_mode,
            "theta_radius": THETA_RADIUS,
        }
    elif name == "CP-spline":
        candidates = []
        for rank in range(1, 129):
            model_i = CPRegressor(p, rank=rank)
            candidates.append((rank, count_parameters(model_i)))
        rank, _ = choose_nearest(candidates, budget)
        model = CPRegressor(p, rank=rank)
        config = {"rank": rank}
    elif name == "TT-spline":
        ranks = matched_tt_ranks(p, N_BASIS, budget)
        model = TTRegressor(p, ranks=ranks)
        config = {"ranks": ranks}
    elif name == "Spline-KAN":
        candidates = []
        for hidden in range(1, 257):
            model_i = SplineKAN(p, hidden=hidden)
            candidates.append((hidden, count_parameters(model_i)))
        hidden, _ = choose_nearest(candidates, budget)
        model = SplineKAN(p, hidden=hidden)
        config = {"hidden": hidden}
    elif name == "MLP":
        candidates = []
        for width in range(2, 257):
            model_i = MLPRegressor(p, width=width)
            candidates.append((width, count_parameters(model_i)))
        width, _ = choose_nearest(candidates, budget)
        model = MLPRegressor(p, width=width)
        config = {"width": width}
    elif name == "ResNet":
        candidates = []
        for n_blocks in [1, 2, 3]:
            for width in range(2, 129):
                model_i = ResNetRegressor(p, width=width, n_blocks=n_blocks)
                candidates.append(((width, n_blocks), count_parameters(model_i)))
        (width, n_blocks), _ = choose_nearest(candidates, budget)
        model = ResNetRegressor(p, width=width, n_blocks=n_blocks)
        config = {"width": width, "n_blocks": n_blocks}
    else:
        raise KeyError(name)
    actual = count_parameters(model)
    return model, config, actual


def match_catboost_budget(budget):
    candidates = []
    for depth in range(2, 8):
        per_tree = (2 ** depth) + depth  # feuilles + seuils d'un arbre symétrique
        iterations = max(1, int(round((budget - 1) / per_tree)))
        for it in {max(1, iterations - 1), iterations, iterations + 1}:
            structural = 1 + it * per_tree
            candidates.append(((depth, it), structural))
    (depth, iterations), structural = choose_nearest(candidates, budget)
    return {"depth": depth, "iterations": iterations}, structural


def parameter_matching_table(p):
    budget = acg_budget(p)
    rows = []
    for name in MODEL_NAMES:
        if name == "CatBoost":
            config, actual = match_catboost_budget(budget)
        else:
            model, config, actual = build_budget_matched_model(name, p, budget)
            del model
        rows.append({
            "model": name, "budget": budget, "active_parameters": actual,
            "relative_gap": (actual - budget) / budget,
            "configuration": json.dumps(config),
        })
    return pd.DataFrame(rows)


display(parameter_matching_table(p=10))

## 7. Trois optimiseurs ACG et protocole équitable

### ACG-BCD-Ball (méthode principale)

Le prédicteur est affine en chacun des blocs

\[
Q_1,\ldots,Q_p,\quad \Theta_1,\ldots,\Theta_{p-1},\quad(\rho,\alpha)
\]

lorsque les autres blocs sont fixés. Chaque $Q_k$ et le readout sont donc mis à
jour par moindres carrés ridge sur tout le fold d'entraînement, en accumulant les
équations normales par chunks GPU. Chaque $\Theta_s$ minimise son sous-problème
quadratique par gradient projeté sur le produit de boules. Ce découpage remplace
l'ancien pseudo-BCD qui optimisait simultanément tout $Q$ et tout $\Theta$.

### ACG-GD-Free

AdamW stochastique met à jour conjointement tous les paramètres ; $\Theta$ est
libre et seule une coupure globale du gradient est appliquée.

### ACG-SBG-Riemannian

Un cycle met à jour une fois chacun des trois blocs grossiers $(Q,\Theta,\rho)$.
Le gradient de $\Theta$ est projeté sur l'espace tangent puis rétracté sur le
produit de sphères. Le budget est exprimé en **cycles complets**, et non en nombre
brut de mises à jour : chaque bloc reçoit donc le même nombre d'occasions
d'apprentissage, indépendamment du nombre de blocs du modèle.

CatBoost conserve son boosting natif ; cette exception est enregistrée.

In [ ]:
try:
    from pynvml import (
        nvmlInit, nvmlDeviceGetHandleByIndex, nvmlDeviceGetMemoryInfo,
    )
    nvmlInit()
    _NVML_HANDLE = nvmlDeviceGetHandleByIndex(0)
except Exception:
    _NVML_HANDLE = None


class GPUMemoryMonitor:
    def __init__(self, interval=0.02):
        self.interval = interval
        self.peak = 0
        self.baseline = 0
        self.stop_event = threading.Event()
        self.thread = None

    def _used(self):
        if _NVML_HANDLE is None:
            return 0
        return int(nvmlDeviceGetMemoryInfo(_NVML_HANDLE).used)

    def _run(self):
        while not self.stop_event.is_set():
            self.peak = max(self.peak, self._used())
            self.stop_event.wait(self.interval)

    def __enter__(self):
        self.baseline = self._used()
        self.peak = self.baseline
        self.stop_event.clear()
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, exc_type, exc, tb):
        self.stop_event.set()
        if self.thread is not None:
            self.thread.join(timeout=1.0)

    @property
    def peak_delta_mb(self):
        return max(0, self.peak - self.baseline) / (1024 ** 2)


@torch.no_grad()
def batched_predict(model, x, batch_size=4096):
    model.eval()
    outputs = []
    for start in range(0, len(x), batch_size):
        outputs.append(model(x[start:start + batch_size]).float().cpu())
    return torch.cat(outputs).numpy()


def _state_dict_cpu(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def _validation_mse(model, x_val, y_val):
    pred = batched_predict(model, x_val)
    return float(np.mean((pred - y_val.detach().cpu().numpy()) ** 2))


def _theta_diagnostics(model):
    if not isinstance(model, ACGRegressor):
        return {
            "theta_norm_mean": float("nan"),
            "theta_norm_min": float("nan"),
            "theta_norm_max": float("nan"),
        }
    norms = model.theta.detach().norm(dim=-1)
    return {
        "theta_norm_mean": float(norms.mean()),
        "theta_norm_min": float(norms.min()),
        "theta_norm_max": float(norms.max()),
    }


def fit_stochastic_block_gradient(
    model: nn.Module,
    x_fit: torch.Tensor,
    y_fit: torch.Tensor,
    x_val: torch.Tensor,
    y_val: torch.Tensor,
    seed: int,
):
    '''Cyclic stochastic block-gradient; sphere blocks use Riemannian updates.'''
    seed_everything(seed)
    model = model.to(DEVICE)
    run_model = model
    if USE_TORCH_COMPILE:
        run_model = torch.compile(model, mode=TORCH_COMPILE_MODE, fullgraph=False)

    blocks = model.parameter_blocks()
    optimizers = {}
    for block in blocks:
        if block.geometry == "euclidean":
            optimizers[block.name] = torch.optim.AdamW(
                block.params, lr=LR_EUCLIDEAN, weight_decay=WEIGHT_DECAY
            )

    best_val = float("inf")
    best_state = _state_dict_cpu(model)
    bad_evals = 0
    history = []
    generator = torch.Generator(device=DEVICE).manual_seed(100_000 + seed)
    all_params = list(model.parameters())
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    with GPUMemoryMonitor() as mem:
        t0 = time.perf_counter()
        cycles_run = 0
        block_updates = 0
        stop = False
        for cycle in range(CFG["gradient_cycles"]):
            for block in blocks:
                active_ids = {id(p) for p in block.params}
                for parameter in all_params:
                    parameter.requires_grad_(id(parameter) in active_ids)

                idx = torch.randint(
                    0, len(x_fit),
                    (min(CFG["batch_size"], len(x_fit)),),
                    generator=generator, device=DEVICE,
                )
                xb, yb = x_fit[idx], y_fit[idx]
                model.train()
                for parameter in block.params:
                    parameter.grad = None
                pred = run_model(xb)
                loss = F.mse_loss(pred.float(), yb.float())
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Perte non finie au bloc {block.name}")
                loss.backward()

                if block.geometry == "sphere":
                    theta = block.params[0]
                    with torch.no_grad():
                        grad_r = riemannian_sphere_gradient(theta, theta.grad, THETA_RADIUS)
                        grad_norm = grad_r.norm().clamp_min(EPS)
                        grad_r.mul_(min(1.0, GRAD_CLIP / float(grad_norm)))
                        theta.add_(grad_r, alpha=-LR_THETA)
                        retract_product_sphere_(theta, THETA_RADIUS)
                        theta.grad = None
                else:
                    torch.nn.utils.clip_grad_norm_(block.params, GRAD_CLIP)
                    optimizers[block.name].step()
                    optimizers[block.name].zero_grad(set_to_none=True)
                block_updates += 1

            cycles_run = cycle + 1
            if cycles_run % CFG["eval_every_cycles"] == 0 or cycles_run == CFG["gradient_cycles"]:
                val_mse = _validation_mse(run_model, x_val, y_val)
                history.append((cycles_run, float(loss.detach().cpu()), val_mse, "cycle"))
                if val_mse < best_val - 1e-7:
                    best_val = val_mse
                    best_state = _state_dict_cpu(model)
                    bad_evals = 0
                else:
                    bad_evals += 1
                    if bad_evals >= CFG["patience"]:
                        stop = True
                        break
            if stop:
                break

        torch.cuda.synchronize()
        train_seconds = time.perf_counter() - t0

    model.load_state_dict(best_state)
    model.to(DEVICE)
    for parameter in model.parameters():
        parameter.requires_grad_(True)
    info = {
        "optimizer": "cyclic-stochastic-block-gradient",
        "cycles_run": cycles_run,
        "block_updates": block_updates,
        "best_val_mse": best_val,
        "train_seconds": train_seconds,
        "peak_gpu_mb_nvml": mem.peak_delta_mb,
        "peak_gpu_mb_torch": torch.cuda.max_memory_allocated() / (1024 ** 2),
        "history": history,
    }
    info.update(_theta_diagnostics(model))
    return model, info


def fit_joint_gradient_free_theta(model, x_fit, y_fit, x_val, y_val, seed):
    '''Joint stochastic AdamW; theta is deliberately unconstrained.'''
    if not isinstance(model, ACGRegressor) or model.theta_mode != "free":
        raise TypeError("fit_joint_gradient_free_theta exige ACG(theta_mode='free')")
    seed_everything(seed)
    model = model.to(DEVICE)
    run_model = (
        torch.compile(model, mode=TORCH_COMPILE_MODE, fullgraph=False)
        if USE_TORCH_COMPILE else model
    )
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR_EUCLIDEAN, weight_decay=WEIGHT_DECAY
    )
    generator = torch.Generator(device=DEVICE).manual_seed(200_000 + seed)
    best_val = float("inf")
    best_state = _state_dict_cpu(model)
    bad_evals, history = 0, []
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    with GPUMemoryMonitor() as mem:
        t0 = time.perf_counter()
        cycles_run = 0
        for cycle in range(CFG["gradient_cycles"]):
            idx = torch.randint(
                0, len(x_fit), (min(CFG["batch_size"], len(x_fit)),),
                generator=generator, device=DEVICE,
            )
            optimizer.zero_grad(set_to_none=True)
            loss = F.mse_loss(run_model(x_fit[idx]).float(), y_fit[idx].float())
            if not torch.isfinite(loss):
                raise FloatingPointError("Perte non finie pour ACG-GD-Free")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            cycles_run = cycle + 1
            if cycles_run % CFG["eval_every_cycles"] == 0 or cycles_run == CFG["gradient_cycles"]:
                val_mse = _validation_mse(run_model, x_val, y_val)
                history.append((cycles_run, float(loss.detach().cpu()), val_mse, "joint"))
                if val_mse < best_val - 1e-7:
                    best_val, best_state, bad_evals = val_mse, _state_dict_cpu(model), 0
                else:
                    bad_evals += 1
                    if bad_evals >= CFG["patience"]:
                        break
        torch.cuda.synchronize()
        train_seconds = time.perf_counter() - t0
    model.load_state_dict(best_state)
    info = {
        "optimizer": "joint-AdamW-free-theta",
        "cycles_run": cycles_run,
        "block_updates": cycles_run,
        "best_val_mse": best_val,
        "train_seconds": train_seconds,
        "peak_gpu_mb_nvml": mem.peak_delta_mb,
        "peak_gpu_mb_torch": torch.cuda.max_memory_allocated() / (1024 ** 2),
        "history": history,
    }
    info.update(_theta_diagnostics(model))
    return model, info


def _normal_equations(model, x, y, design_fn, old, ridge=BCD_RIDGE, prox=BCD_PROX):
    '''Accumulate X'X and X'y by chunks without materialising the full design.'''
    dim = old.numel()
    gram = torch.zeros(dim, dim, device=DEVICE, dtype=torch.float32)
    rhs = torch.zeros(dim, device=DEVICE, dtype=torch.float32)
    n_total = 0
    for start in range(0, len(x), CFG["bcd_chunk_size"]):
        xb = x[start:start + CFG["bcd_chunk_size"]]
        yb = y[start:start + CFG["bcd_chunk_size"]]
        X, offset = design_fn(model, xb)
        target = yb.float() - offset.float()
        X = X.float()
        gram.add_(X.T @ X)
        rhs.add_(X.T @ target)
        n_total += len(xb)
    gram.div_(max(n_total, 1))
    rhs.div_(max(n_total, 1))
    eye = torch.eye(dim, device=DEVICE, dtype=gram.dtype)
    system = gram + (ridge + prox) * eye
    rhs = rhs + prox * old.detach().flatten().float()
    return system, rhs


def _solve_spd(system, rhs):
    chol, flag = torch.linalg.cholesky_ex(system)
    if int(flag.max()) == 0:
        return torch.cholesky_solve(rhs[:, None], chol).squeeze(1)
    return torch.linalg.lstsq(system, rhs[:, None]).solution.squeeze(1)


def _q_design(model, basis, k):
    with torch.enable_grad():
        lifts = torch.einsum("bpn,rpnm->brpm", basis, model.q.detach()).detach()
        lifts.requires_grad_(True)
        state = model.compose_lifts(lifts, theta=model.theta.detach())
        pred = model.alpha.detach() + torch.einsum("brm,rm->b", state, model.rho.detach())
        grad = torch.autograd.grad(pred.sum(), lifts, create_graph=False)[0][:, :, k]
        X = torch.einsum("bn,brm->brnm", basis[:, k], grad).reshape(len(basis), -1)
        current = model.q[:, k].detach().reshape(-1)
        offset = pred.detach() - X.detach() @ current
    return X.detach(), offset


def _theta_design(model, basis, s):
    with torch.enable_grad():
        lifts = torch.einsum("bpn,rpnm->brpm", basis, model.q.detach()).detach()
        before = lifts[:, :, 0]
        for t in range(s):
            before = theta_product(
                before, lifts[:, :, t + 1], model.theta[t].detach(),
                model.pair_a, model.pair_b,
            )
        after = theta_product(
            before, lifts[:, :, s + 1], model.theta[s].detach(),
            model.pair_a, model.pair_b,
        ).detach().requires_grad_(True)
        state = after
        for t in range(s + 1, model.p - 1):
            state = theta_product(
                state, lifts[:, :, t + 1], model.theta[t].detach(),
                model.pair_a, model.pair_b,
            )
        pred = model.alpha.detach() + torch.einsum("brm,rm->b", state, model.rho.detach())
        grad_after = torch.autograd.grad(pred.sum(), after, create_graph=False)[0]
        xv, yv = before[..., 1:], lifts[:, :, s + 1, 1:]
        wedge = xv[..., model.pair_a] * yv[..., model.pair_b]
        wedge = wedge - xv[..., model.pair_b] * yv[..., model.pair_a]
        X = torch.einsum("brh,brd->bhd", wedge, grad_after[..., 1:]).reshape(len(basis), -1)
        current = model.theta[s].detach().reshape(-1)
        offset = pred.detach() - X.detach() @ current
    return X.detach(), offset


def _readout_design(model, basis):
    with torch.no_grad():
        lifts = torch.einsum("bpn,rpnm->brpm", basis, model.q)
        state = model.compose_lifts(lifts)
        X = torch.cat([state.reshape(len(basis), -1), torch.ones(len(basis), 1, device=DEVICE)], 1)
        offset = torch.zeros(len(basis), device=DEVICE)
    return X, offset


def _projected_theta_solve(system, rhs, current, radius, steps):
    w = current.detach().flatten().float().clone()
    lipschitz = torch.linalg.eigvalsh(system).amax().clamp_min(1e-8)
    for _ in range(steps):
        w = w - (system @ w - rhs) / lipschitz
        rows = w.view_as(current)
        norms = rows.norm(dim=-1, keepdim=True).clamp_min(EPS)
        rows.mul_(torch.clamp(radius / norms, max=1.0))
        w = rows.reshape(-1)
    return w


def fit_coordinate_bcd_ball(model, x_fit, y_fit, x_val, y_val, seed):
    '''True coordinate blocks with affine least-squares subproblems.'''
    if not isinstance(model, ACGRegressor) or model.theta_mode != "ball":
        raise TypeError("fit_coordinate_bcd_ball exige ACG(theta_mode='ball')")
    seed_everything(seed)
    model = model.to(DEVICE)
    project_product_ball_(model.theta, THETA_RADIUS)
    best_val = _validation_mse(model, x_val, y_val)
    best_state = _state_dict_cpu(model)
    bad_evals, history = 0, [(0, float("nan"), best_val, "initial")]
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    with GPUMemoryMonitor() as mem:
        t0 = time.perf_counter()
        block_updates = 0
        cycles_run = 0
        for cycle in range(CFG["bcd_cycles"]):
            model.train()
            for k in range(model.p):
                old = model.q[:, k].detach()
                system, rhs = _normal_equations(
                    model, x_fit, y_fit, lambda mod, xb, kk=k: _q_design(mod, xb, kk), old
                )
                solution = _solve_spd(system, rhs).view_as(old)
                with torch.no_grad():
                    model.q[:, k].copy_(solution)
                block_updates += 1

            for s in range(model.p - 1):
                old = model.theta[s].detach()
                system, rhs = _normal_equations(
                    model, x_fit, y_fit,
                    lambda mod, xb, ss=s: _theta_design(mod, xb, ss), old,
                )
                solution = _projected_theta_solve(
                    system, rhs, old, THETA_RADIUS, CFG["theta_inner_steps"]
                ).view_as(old)
                with torch.no_grad():
                    model.theta[s].copy_(solution)
                block_updates += 1

            old = torch.cat([model.rho.detach().reshape(-1), model.alpha.detach()[None]])
            system, rhs = _normal_equations(
                model, x_fit, y_fit, _readout_design, old, ridge=BCD_RIDGE, prox=0.0
            )
            # L'intercept n'est pas régularisé.
            system[-1, -1].sub_(BCD_RIDGE)
            solution = _solve_spd(system, rhs)
            with torch.no_grad():
                model.rho.copy_(solution[:-1].view_as(model.rho))
                model.alpha.copy_(solution[-1])
                model.gauge_balance_()
                project_product_ball_(model.theta, THETA_RADIUS)
            block_updates += 1
            cycles_run = cycle + 1

            bcd_eval_every = max(1, CFG["eval_every_cycles"] // 5)
            if cycles_run % bcd_eval_every == 0 or cycles_run == CFG["bcd_cycles"]:
                val_mse = _validation_mse(model, x_val, y_val)
                history.append((cycles_run, float("nan"), val_mse, "full-coordinate-cycle"))
                if val_mse < best_val - 1e-7:
                    best_val, best_state, bad_evals = val_mse, _state_dict_cpu(model), 0
                else:
                    bad_evals += 1
                    if bad_evals >= CFG["patience"]:
                        break
        torch.cuda.synchronize()
        train_seconds = time.perf_counter() - t0
    model.load_state_dict(best_state)
    project_product_ball_(model.theta, THETA_RADIUS)
    info = {
        "optimizer": "coordinate-BCD-ridge-product-ball",
        "cycles_run": cycles_run,
        "block_updates": block_updates,
        "best_val_mse": best_val,
        "train_seconds": train_seconds,
        "peak_gpu_mb_nvml": mem.peak_delta_mb,
        "peak_gpu_mb_torch": torch.cuda.max_memory_allocated() / (1024 ** 2),
        "history": history,
    }
    info.update(_theta_diagnostics(model))
    return model, info


def fit_catboost(x_fit, y_fit, x_val, y_val, seed, budget):
    config, structural_params = match_catboost_budget(budget)
    common = dict(
        iterations=config["iterations"], depth=config["depth"],
        learning_rate=0.05, loss_function="RMSE", l2_leaf_reg=3.0,
        random_seed=seed, bootstrap_type="Bernoulli", subsample=0.9,
        allow_writing_files=False, verbose=False,
    )
    model = CatBoostRegressor(**common, task_type="GPU", devices="0")
    torch.cuda.synchronize()
    with GPUMemoryMonitor() as mem:
        t0 = time.perf_counter()
        backend = "GPU"
        try:
            model.fit(
                x_fit, y_fit, eval_set=(x_val, y_val),
                use_best_model=False, verbose=False,
            )
        except Exception as gpu_exc:
            print("  CatBoost GPU indisponible, repli CPU:", str(gpu_exc)[:160])
            backend = "CPU-fallback"
            model = CatBoostRegressor(**common, task_type="CPU", thread_count=-1)
            model.fit(
                x_fit, y_fit, eval_set=(x_val, y_val),
                use_best_model=False, verbose=False,
            )
        train_seconds = time.perf_counter() - t0
    val_pred = model.predict(x_val)
    return model, config, structural_params, {
        "optimizer": f"native-symmetric-gradient-boosting-{backend}",
        "cycles_run": config["iterations"],
        "block_updates": config["iterations"],
        "best_val_mse": float(np.mean((val_pred - y_val) ** 2)),
        "train_seconds": train_seconds,
        "peak_gpu_mb_nvml": mem.peak_delta_mb,
        "peak_gpu_mb_torch": float("nan"),
        "history": [],
        "theta_norm_mean": float("nan"),
        "theta_norm_min": float("nan"),
        "theta_norm_max": float("nan"),
    }

## 8. Génération des quatre jeux synthétiques

Les cibles ACG, CP et TT sont générées par les familles correspondantes avec un bruit de 2 %. Friedman-1 apporte une cible indépendante de ces trois formats.

In [ ]:
@torch.no_grad()
def make_structured_synthetic(kind: str, n=SYNTHETIC_N, seed=2026):
    seed_everything(seed)
    p = 10 if kind == "Friedman-1" else 6
    x = torch.rand(n, p)
    if kind == "Friedman-1":
        y = (
            10.0 * torch.sin(math.pi * x[:, 0] * x[:, 1])
            + 20.0 * (x[:, 2] - 0.5).square()
            + 10.0 * x[:, 3] + 5.0 * x[:, 4]
        )
    else:
        basis = feature_basis(x)
        if kind == "ACG-structured":
            model = ACGRegressor(p, theta_mode="ball")
            model.q.normal_(0.0, 0.22)
            model.q[..., 0].add_(0.9)
            model.rho.normal_(0.0, 0.35)
            model.theta.mul_(0.65).add_(0.10 * torch.randn_like(model.theta))
            project_product_ball_(model.theta, THETA_RADIUS)
        elif kind == "CP-structured":
            model = CPRegressor(p, rank=3)
            model.a.normal_(1.0, 0.18)
            model.lam.normal_(0.0, 0.6)
        elif kind == "TT-structured":
            model = TTRegressor(p, ranks=[1] + [3] * (p - 1) + [1])
        else:
            raise KeyError(kind)
        y = model(basis).float()
    noise = 0.02 * y.std(unbiased=False) * torch.randn_like(y)
    y = y + noise
    X = pd.DataFrame(x.numpy(), columns=[f"x{k+1}" for k in range(p)])
    return TabularDataset(kind, "synthetic", X, y.numpy().astype(np.float32))


def build_registry():
    registry = {name: load_real_dataset(name) for name in REAL_DATASETS}
    registry.update({name: make_structured_synthetic(name) for name in SYNTHETIC_DATASETS})
    return registry


DATASETS = build_registry()
catalog = pd.DataFrame([
    {
        "dataset": ds.name, "kind": ds.kind, "samples": len(ds.X),
        "raw_features": ds.X.shape[1],
        "categorical_features": int(sum(not pd.api.types.is_numeric_dtype(ds.X[c]) for c in ds.X)),
    }
    for ds in DATASETS.values()
])
display(catalog)

## 9. Contrôles structurels avant les expériences

Cette cellule vérifie les dimensions, l'égalité exacte des paramètres entre les
trois ACG, les contraintes boule/sphère, la tangence du gradient riemannien et
l'identité affine de chaque vrai bloc BCD avant les calculs longs.

In [ ]:
def structural_checks():
    seed_everything(0)
    p = 6
    x = torch.rand(32, p, device=DEVICE)
    basis = feature_basis(x)
    budget = acg_budget(p)
    rows = []
    for name in MODEL_NAMES[:-1]:
        model, config, params = build_budget_matched_model(name, p, budget)
        model = model.to(DEVICE)
        inp = basis if model.input_kind == "basis" else x
        out = model(inp)
        assert out.shape == (len(x),)
        assert torch.isfinite(out).all()
        assert len(model.parameter_blocks()) >= 2
        if name == "ACG-BCD-Ball":
            norms = model.theta.norm(dim=-1)
            assert bool((norms <= THETA_RADIUS + 1e-6).all())
        elif name == "ACG-SBG-Riemannian":
            norms = model.theta.norm(dim=-1)
            assert torch.allclose(norms, torch.full_like(norms, THETA_RADIUS), atol=1e-5)
            g = torch.randn_like(model.theta)
            rg = riemannian_sphere_gradient(model.theta, g, THETA_RADIUS)
            assert float((rg * model.theta).sum(-1).abs().max()) < 2e-5
        rows.append({
            "model": name, "parameters": params,
            "gap_%": 100 * (params - budget) / budget,
            "config": config,
        })
    cb_config, cb_params = match_catboost_budget(budget)
    rows.append({
        "model": "CatBoost", "parameters": cb_params,
        "gap_%": 100 * (cb_params - budget) / budget,
        "config": cb_config,
    })
    table = pd.DataFrame(rows)
    acg_counts = table[table.model.isin(ACG_VARIANTS)].parameters.unique()
    assert len(acg_counts) == 1 and int(acg_counts[0]) == budget

    # Identités f = offset + X w pour les blocs Q_k, Theta_s et readout.
    acg = ACGRegressor(p, theta_mode="ball").to(DEVICE)
    pred = acg(basis).detach()
    max_errors = []
    for k in [0, p - 1]:
        X, offset = _q_design(acg, basis, k)
        recon = offset + X @ acg.q[:, k].detach().reshape(-1)
        max_errors.append(float((pred - recon).abs().max()))
    for s in [0, p - 2]:
        X, offset = _theta_design(acg, basis, s)
        recon = offset + X @ acg.theta[s].detach().reshape(-1)
        max_errors.append(float((pred - recon).abs().max()))
    X, offset = _readout_design(acg, basis)
    readout = torch.cat([acg.rho.detach().reshape(-1), acg.alpha.detach()[None]])
    max_errors.append(float((pred - (offset + X @ readout)).abs().max()))
    assert max(max_errors) < 2e-4, max_errors
    print("Erreur affine maximale des blocs BCD :", max(max_errors))

    # Les solveurs doivent diminuer leur sous-problème quadratique régularisé.
    y_check = pred + 0.05 * torch.randn_like(pred)
    old_q = acg.q[:, 0].detach()
    system_q, rhs_q = _normal_equations(
        acg, basis, y_check, lambda mod, xb: _q_design(mod, xb, 0), old_q
    )
    new_q = _solve_spd(system_q, rhs_q)
    quad = lambda A, b, w: 0.5 * w @ A @ w - b @ w
    assert float(quad(system_q, rhs_q, new_q)) <= float(
        quad(system_q, rhs_q, old_q.reshape(-1))
    ) + 1e-5

    old_t = acg.theta[0].detach()
    system_t, rhs_t = _normal_equations(
        acg, basis, y_check, lambda mod, xb: _theta_design(mod, xb, 0), old_t
    )
    new_t = _projected_theta_solve(
        system_t, rhs_t, old_t, THETA_RADIUS, max(20, CFG["theta_inner_steps"])
    )
    assert float(quad(system_t, rhs_t, new_t)) <= float(
        quad(system_t, rhs_t, old_t.reshape(-1))
    ) + 1e-5
    assert float(new_t.view_as(old_t).norm(dim=-1).max()) <= THETA_RADIUS + 1e-6
    return table


CHECKS = structural_checks()
display(CHECKS)
assert np.isfinite(CHECKS["parameters"]).all()
print("Contrôles structurels réussis.")

## 10. Boucle expérimentale avec reprise

Les résultats sont écrits après chaque entraînement. Une relance du notebook ignore automatiquement les clés `(dataset, fold, seed, model)` déjà terminées.

In [ ]:
def atomic_csv(df: pd.DataFrame, path: Path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def selected_dataset_names():
    real = REAL_DATASETS[:CFG["real_limit"]] if CFG["real_limit"] else REAL_DATASETS
    synth = (
        SYNTHETIC_DATASETS[:CFG["synthetic_limit"]]
        if CFG["synthetic_limit"] else SYNTHETIC_DATASETS
    )
    return real + synth


def run_key(row):
    return (str(row["dataset"]), int(row["fold"]), int(row["seed"]), str(row["model"]))


def run_benchmark():
    results = pd.read_csv(RAW_PATH).to_dict("records") if RESUME and RAW_PATH.exists() else []
    errors = pd.read_csv(ERROR_PATH).to_dict("records") if ERROR_PATH.exists() else []
    histories = (
        pd.read_csv(HISTORY_PATH).to_dict("records")
        if RESUME and HISTORY_PATH.exists() else []
    )
    completed = {run_key(row) for row in results if row.get("status", "ok") == "ok"}
    names = selected_dataset_names()
    total = len(names) * CFG["folds"] * len(CFG["seeds"]) * len(MODEL_NAMES)
    print(f"Exécutions prévues : {total}; déjà terminées : {len(completed)}")

    for dataset_name in names:
        dataset = DATASETS[dataset_name]
        kfold = KFold(n_splits=CFG["folds"], shuffle=True, random_state=2026)
        folds = list(kfold.split(np.arange(len(dataset.X))))
        for fold_id, (train_idx, test_idx) in enumerate(folds):
            for seed in CFG["seeds"]:
                # Le split interne est fixé par le fold : les graines ne changent
                # que l'initialisation/ordre stochastique, jamais les observations.
                prepared = prepare_fold(dataset, train_idx, test_idx, fold_id)
                p = prepared.x_fit.shape[1]
                budget = acg_budget(p)
                fold_hash = hashlib.sha256(
                    np.asarray(test_idx, dtype=np.int64).tobytes()
                ).hexdigest()[:16]

                x_fit_raw = torch.as_tensor(prepared.x_fit, device=DEVICE)
                x_val_raw = torch.as_tensor(prepared.x_val, device=DEVICE)
                x_test_raw = torch.as_tensor(prepared.x_test, device=DEVICE)
                y_fit_t = torch.as_tensor(prepared.y_fit, device=DEVICE)
                y_val_t = torch.as_tensor(prepared.y_val, device=DEVICE)
                x_fit_basis = feature_basis(x_fit_raw)
                x_val_basis = feature_basis(x_val_raw)
                x_test_basis = feature_basis(x_test_raw)

                for model_name in MODEL_NAMES:
                    key = (dataset_name, fold_id, seed, model_name)
                    if key in completed:
                        continue
                    print(
                        f"[{dataset_name:18s}] fold={fold_id+1}/{CFG['folds']} "
                        f"seed={seed} model={model_name:10s} p={p} budget={budget}"
                    )
                    try:
                        seed_everything(seed)
                        if model_name == "CatBoost":
                            model, config, actual_params, info = fit_catboost(
                                prepared.x_fit, prepared.y_fit,
                                prepared.x_val, prepared.y_val,
                                seed, budget,
                            )
                            pred_scaled = model.predict(prepared.x_test)
                            del model
                        else:
                            model, config, actual_params = build_budget_matched_model(
                                model_name, p, budget
                            )
                            if model.input_kind == "basis":
                                tr_in, va_in, te_in = x_fit_basis, x_val_basis, x_test_basis
                            else:
                                tr_in, va_in, te_in = x_fit_raw, x_val_raw, x_test_raw
                            if model_name == "ACG-BCD-Ball":
                                model, info = fit_coordinate_bcd_ball(
                                    model, tr_in, y_fit_t, va_in, y_val_t, seed
                                )
                            elif model_name == "ACG-GD-Free":
                                model, info = fit_joint_gradient_free_theta(
                                    model, tr_in, y_fit_t, va_in, y_val_t, seed
                                )
                            else:
                                model, info = fit_stochastic_block_gradient(
                                    model, tr_in, y_fit_t, va_in, y_val_t, seed
                                )
                            pred_scaled = batched_predict(model, te_in)
                            del model

                        pred = pred_scaled * prepared.y_std + prepared.y_mean
                        metrics = regression_metrics(prepared.y_test, pred)
                        row = {
                            "status": "ok", "dataset": dataset_name, "kind": dataset.kind,
                            "fold": fold_id, "seed": seed, "model": model_name,
                            "test_fold_sha256_16": fold_hash,
                            "samples": len(dataset.X), "features_after_encoding": p,
                            "parameter_budget": budget, "active_parameters": actual_params,
                            "parameter_gap": actual_params - budget,
                            "relative_parameter_gap": (actual_params - budget) / budget,
                            "configuration": json.dumps(config),
                            "analysis_role": (
                                "confirmatory" if model_name in CONFIRMATORY_MODELS
                                else "optimizer_ablation"
                            ),
                            **metrics,
                            **{k: info[k] for k in [
                                "optimizer", "cycles_run", "block_updates", "best_val_mse",
                                "train_seconds", "peak_gpu_mb_nvml", "peak_gpu_mb_torch",
                                "theta_norm_mean", "theta_norm_min", "theta_norm_max",
                            ]},
                        }
                        results.append(row)
                        for cycle_i, train_loss, val_loss, block_name in info["history"]:
                            histories.append({
                                "dataset": dataset_name, "kind": dataset.kind,
                                "fold": fold_id, "seed": seed, "model": model_name,
                                "cycle": cycle_i, "train_batch_mse": train_loss,
                                "validation_mse": val_loss, "stage": block_name,
                            })
                        completed.add(key)
                        atomic_csv(pd.DataFrame(results), RAW_PATH)
                        if histories:
                            atomic_csv(pd.DataFrame(histories), HISTORY_PATH)
                    except Exception as exc:
                        errors.append({
                            "dataset": dataset_name, "fold": fold_id, "seed": seed,
                            "model": model_name, "error_type": type(exc).__name__,
                            "message": str(exc)[:1000],
                        })
                        atomic_csv(pd.DataFrame(errors), ERROR_PATH)
                        print("  ERREUR:", type(exc).__name__, str(exc)[:250])
                    finally:
                        gc.collect()
                        torch.cuda.empty_cache()

                del x_fit_raw, x_val_raw, x_test_raw, y_fit_t, y_val_t
                del x_fit_basis, x_val_basis, x_test_basis
                gc.collect()
                torch.cuda.empty_cache()
    return pd.DataFrame(results), pd.DataFrame(errors), pd.DataFrame(histories)


RAW_RESULTS, ERRORS, OPTIMIZATION_HISTORY = run_benchmark()
print("Résultats valides :", len(RAW_RESULTS))
print("Erreurs            :", len(ERRORS))
print("Points de convergence :", len(OPTIMIZATION_HISTORY))
display(RAW_RESULTS.tail())

## 11. Agrégation, rangs et compromis précision–complexité

In [ ]:
def aggregate_results(raw):
    ok = raw[raw.status == "ok"].copy()
    summary = (
        ok.groupby(["kind", "dataset", "model"], as_index=False)
        .agg(
            nrmse_mean=("nrmse", "mean"), nrmse_std=("nrmse", "std"),
            r2_mean=("r2", "mean"), r2_std=("r2", "std"),
            parameters=("active_parameters", "median"),
            parameter_gap=("relative_parameter_gap", "median"),
            train_seconds=("train_seconds", "mean"),
            peak_gpu_mb=("peak_gpu_mb_nvml", "mean"),
            repetitions=("nrmse", "size"),
        )
    )
    summary["rank_all_nine"] = summary.groupby("dataset")["nrmse_mean"].rank(method="average")
    summary["rank"] = np.nan
    confirmatory_mask = summary.model.isin(CONFIRMATORY_MODELS)
    summary.loc[confirmatory_mask, "rank"] = (
        summary.loc[confirmatory_mask].groupby("dataset")["nrmse_mean"].rank(method="average")
    )
    model_summary = (
        summary[confirmatory_mask].groupby(["kind", "model"], as_index=False)
        .agg(
            average_rank=("rank", "mean"),
            mean_nrmse=("nrmse_mean", "mean"),
            median_nrmse=("nrmse_mean", "median"),
            median_parameters=("parameters", "median"),
            mean_time=("train_seconds", "mean"),
            mean_peak_gpu_mb=("peak_gpu_mb", "mean"),
            datasets=("dataset", "nunique"),
        )
        .sort_values(["kind", "average_rank"])
    )
    optimizer_summary = (
        summary[summary.model.isin(ACG_VARIANTS)]
        .groupby(["kind", "model"], as_index=False)
        .agg(
            optimizer_average_rank=("rank_all_nine", "mean"),
            mean_nrmse=("nrmse_mean", "mean"),
            median_nrmse=("nrmse_mean", "median"),
            mean_time=("train_seconds", "mean"),
            mean_peak_gpu_mb=("peak_gpu_mb", "mean"),
            datasets=("dataset", "nunique"),
        )
        .sort_values(["kind", "optimizer_average_rank"])
    )
    return summary, model_summary, optimizer_summary


DATASET_SUMMARY, MODEL_SUMMARY, OPTIMIZER_SUMMARY = aggregate_results(RAW_RESULTS)
DATASET_SUMMARY.to_csv(OUTDIR / "dataset_model_summary.csv", index=False)
MODEL_SUMMARY.to_csv(OUTDIR / "model_summary.csv", index=False)
OPTIMIZER_SUMMARY.to_csv(OUTDIR / "acg_optimizer_ablation.csv", index=False)
print("Comparaison confirmatoire (7 méthodes) :")
display(MODEL_SUMMARY)
print("Ablation d'optimisation ACG (mêmes paramètres) :")
display(OPTIMIZER_SUMMARY)

real_table = DATASET_SUMMARY[
    (DATASET_SUMMARY.kind == "real") & DATASET_SUMMARY.model.isin(CONFIRMATORY_MODELS)
].pivot(
    index="dataset", columns="model", values="nrmse_mean"
)
display(real_table.style.highlight_min(axis=1))

## 12. Friedman, Nemenyi et Wilcoxon–Holm

Les tests confirmatoires utilisent les sept méthodes pré-déclarées :
`ACG-BCD-Ball` et les six baselines. `ACG-GD-Free` et
`ACG-SBG-Riemannian` restent exclus de ces tests et sont analysés séparément comme
ablation. Les folds et initialisations stabilisent l'estimation ; l'unité du
benchmark multi-dataset reste le dataset.

In [ ]:
def statistical_tests(dataset_summary):
    pivot = dataset_summary[dataset_summary.kind == "real"].pivot(
        index="dataset", columns="model", values="nrmse_mean"
    )
    pivot = pivot.reindex(columns=CONFIRMATORY_MODELS).dropna(axis=0, how="any")
    if len(pivot) < 3:
        print("Tests inférentiels ignorés : moins de trois datasets réels complets.")
        neutral = pd.DataFrame(
            np.ones((len(CONFIRMATORY_MODELS), len(CONFIRMATORY_MODELS))),
            index=CONFIRMATORY_MODELS, columns=CONFIRMATORY_MODELS,
        )
        empty = pd.DataFrame(columns=[
            "comparison", "model", "wilcoxon_statistic", "p_raw",
            "median_delta_nrmse_primary_ACG_minus_model", "p_holm", "reject_0.05",
        ])
        info = {
            "datasets_complete": int(len(pivot)),
            "friedman_statistic": None,
            "friedman_pvalue": None,
        }
        return pivot, neutral, empty, info

    stat, p_friedman = friedmanchisquare(
        *[pivot[m].to_numpy() for m in CONFIRMATORY_MODELS]
    )
    nemenyi = sp.posthoc_nemenyi_friedman(pivot.to_numpy())
    nemenyi.index = CONFIRMATORY_MODELS
    nemenyi.columns = CONFIRMATORY_MODELS

    rows = []
    for model in CONFIRMATORY_MODELS:
        if model == PRIMARY_ACG:
            continue
        a, b = pivot[PRIMARY_ACG].to_numpy(), pivot[model].to_numpy()
        if np.allclose(a, b):
            w_stat, p = 0.0, 1.0
        else:
            w_stat, p = wilcoxon(a, b, alternative="two-sided", zero_method="wilcox")
        rows.append({
            "comparison": f"{PRIMARY_ACG} vs {model}", "model": model,
            "wilcoxon_statistic": float(w_stat), "p_raw": float(p),
            "median_delta_nrmse_primary_ACG_minus_model": float(np.median(a - b)),
        })
    pairwise = pd.DataFrame(rows)
    reject, p_holm, _, _ = multipletests(pairwise.p_raw, method="holm")
    pairwise["p_holm"] = p_holm
    pairwise["reject_0.05"] = reject

    test_info = {
        "datasets_complete": int(len(pivot)),
        "friedman_statistic": float(stat),
        "friedman_pvalue": float(p_friedman),
    }
    return pivot, nemenyi, pairwise, test_info


REAL_PIVOT, NEMENYI, WILCOXON_HOLM, TEST_INFO = statistical_tests(DATASET_SUMMARY)
NEMENYI.to_csv(OUTDIR / "nemenyi_pvalues.csv")
WILCOXON_HOLM.to_csv(OUTDIR / "wilcoxon_holm.csv", index=False)
(OUTDIR / "friedman.json").write_text(json.dumps(TEST_INFO, indent=2))
print(TEST_INFO)
display(WILCOXON_HOLM)
display(NEMENYI)

## 13. Figures publication

In [ ]:
sns.set_theme(style="whitegrid", context="paper")

# Heatmap NRMSE par dataset réel
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(REAL_PIVOT, annot=True, fmt=".3f", cmap="viridis_r", ax=ax)
ax.set_title("Mean NRMSE over 5 folds × seeds (lower is better)")
fig.tight_layout()
fig.savefig(OUTDIR / "real_dataset_nrmse_heatmap.pdf", bbox_inches="tight")
fig.savefig(OUTDIR / "real_dataset_nrmse_heatmap.png", dpi=220, bbox_inches="tight")
plt.show()

# Rangs moyens
real_models = MODEL_SUMMARY[MODEL_SUMMARY.kind == "real"].sort_values("average_rank")
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(real_models.model, real_models.average_rank, color="#315f8c")
ax.invert_yaxis()
ax.set_xlabel("Average rank (lower is better)")
ax.set_title("Average rank across real regression datasets")
fig.tight_layout()
fig.savefig(OUTDIR / "average_ranks.pdf", bbox_inches="tight")
fig.savefig(OUTDIR / "average_ranks.png", dpi=220, bbox_inches="tight")
plt.show()

# Ablation des trois optimiseurs ACG, à architecture identique
opt_real = OPTIMIZER_SUMMARY[OPTIMIZER_SUMMARY.kind == "real"].sort_values(
    "optimizer_average_rank"
)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(opt_real.model, opt_real.optimizer_average_rank, color=["#2a9d8f", "#e9c46a", "#e76f51"])
ax.invert_yaxis()
ax.set_xlabel("Average rank among all nine methods (lower is better)")
ax.set_title("ACG optimizer ablation — identical architecture and parameter count")
fig.tight_layout()
fig.savefig(OUTDIR / "acg_optimizer_ablation.pdf", bbox_inches="tight")
fig.savefig(OUTDIR / "acg_optimizer_ablation.png", dpi=220, bbox_inches="tight")
plt.show()

# Convergence sur la cible ACG structure-matched
if not OPTIMIZATION_HISTORY.empty:
    conv = OPTIMIZATION_HISTORY[
        (OPTIMIZATION_HISTORY.dataset == "ACG-structured")
        & OPTIMIZATION_HISTORY.model.isin(ACG_VARIANTS)
    ].copy()
    if len(conv):
        fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
        for ax, name in zip(axes, ACG_VARIANTS):
            part = conv[conv.model == name]
            sns.lineplot(data=part, x="cycle", y="validation_mse", errorbar="sd", ax=ax)
            ax.set_title(name)
            ax.set_yscale("log")
        axes[0].set_ylabel("Validation MSE (standardized target, log scale)")
        fig.suptitle("Optimization diagnostic on ACG-structured target")
        fig.tight_layout()
        fig.savefig(OUTDIR / "acg_optimizer_convergence.pdf", bbox_inches="tight")
        fig.savefig(OUTDIR / "acg_optimizer_convergence.png", dpi=220, bbox_inches="tight")
        plt.show()

# P-values Nemenyi
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(NEMENYI, annot=True, fmt=".3f", cmap="mako_r", vmin=0, vmax=1, ax=ax)
ax.set_title("Nemenyi post-hoc p-values")
fig.tight_layout()
fig.savefig(OUTDIR / "nemenyi_heatmap.pdf", bbox_inches="tight")
fig.savefig(OUTDIR / "nemenyi_heatmap.png", dpi=220, bbox_inches="tight")
plt.show()

# Diagramme de différence critique si disponible
try:
    ranks = DATASET_SUMMARY[
        (DATASET_SUMMARY.kind == "real")
        & DATASET_SUMMARY.model.isin(CONFIRMATORY_MODELS)
    ].groupby("model")["rank"].mean()
    fig, ax = plt.subplots(figsize=(10, 4))
    sp.critical_difference_diagram(ranks, NEMENYI, ax=ax)
    ax.set_title("Critical difference diagram — real datasets")
    fig.tight_layout()
    fig.savefig(OUTDIR / "critical_difference.pdf", bbox_inches="tight")
    fig.savefig(OUTDIR / "critical_difference.png", dpi=220, bbox_inches="tight")
    plt.show()
except Exception as exc:
    print("Diagramme CD non généré:", type(exc).__name__, str(exc)[:180])

## 14. Export reproductible

Le ZIP contient les résultats bruts, résumés, tests, figures, tables LaTeX, configuration et empreinte du dépôt de données.

In [ ]:
DATASET_SUMMARY.to_latex(
    OUTDIR / "dataset_model_summary.tex", index=False, float_format="%.4f"
)
MODEL_SUMMARY.to_latex(
    OUTDIR / "model_summary.tex", index=False, float_format="%.4f"
)
OPTIMIZER_SUMMARY.to_latex(
    OUTDIR / "acg_optimizer_ablation.tex", index=False, float_format="%.4f"
)
WILCOXON_HOLM.to_latex(
    OUTDIR / "wilcoxon_holm.tex", index=False, float_format="%.4g"
)

metadata = {
    "profile": PROFILE,
    "configuration": CFG,
    "real_datasets": REAL_DATASETS,
    "synthetic_datasets": SYNTHETIC_DATASETS,
    "models": MODEL_NAMES,
    "confirmatory_models": CONFIRMATORY_MODELS,
    "acg_optimizer_ablation": ACG_VARIANTS,
    "n_basis": N_BASIS,
    "spline_degree": SPLINE_DEGREE,
    "acg_m": ACG_M,
    "acg_rank": ACG_RANK,
    "theta_domains": {
        "ACG-BCD-Ball": "product of closed row-wise l2 balls",
        "ACG-GD-Free": "unconstrained Euclidean",
        "ACG-SBG-Riemannian": "product of row-wise spheres",
    },
    "theta_radius": THETA_RADIUS,
    "data_repository_commit": DATA_COMMIT,
    "gpu": GPU_NAME,
    "torch": torch.__version__,
    "torch_compile": {
        "enabled_for_gradient_methods": USE_TORCH_COMPILE,
        "mode": TORCH_COMPILE_MODE,
        "coordinate_bcd": "eager (dynamic Jacobian designs and exact block solves)",
    },
    "preprocessing": "train-only MinMax numerical + train-only OneHot categorical",
    "parameter_matching": "nearest active parameter count; CatBoost structural count",
}
(OUTDIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2))

archive_base = str(WORK_ROOT / f"ACG_Optimization_Variants_13Real_4Synthetic_{PROFILE}")
archive = shutil.make_archive(archive_base, "zip", OUTDIR)
print("Archive créée :", archive)

from google.colab import files
files.download(archive)

## Lecture scientifique attendue

Le résultat principal ne doit pas être « ACG gagne partout ». L'analyse doit répondre à quatre questions :

1. le vrai BCD coordonnée avec $\Theta$ dans la boule ferme-t-il l'écart
   d'optimisation sur la cible ACG structure-matched ?
2. la contrainte sphérique aide-t-elle ou empêche-t-elle l'extinction de termes
   antisymétriques inutiles par rapport à $\Theta$ libre ou borné ?
3. `ACG-BCD-Ball` obtient-il un meilleur rang moyen que CP, TT et Spline-KAN à
   budget de paramètres comparable ?
4. le gain éventuel compense-t-il son temps et sa mémoire supplémentaires ?

Les résultats synthétiques et réels doivent être présentés séparément. Les tests
statistiques principaux portent sur les 13 datasets réels et uniquement sur les
sept méthodes confirmatoires pré-déclarées.